In [1]:
# ── User Configuration ────────────────────────────────────────────────────────
symbolName      = "XAUUSD"   # Symbol name (must match broker exactly)
num_chunks      = 52 * 10   # Number of time chunks to fetch
weeks_per_chunk = 1         # Width of each chunk in weeks
period_str      = "M1"      # Bar period: M1 M2 M3 M4 M5 M10 M15 M30 H1 H4 H12 D1 W1 MN1

In [2]:
from ctrader_open_api import Client, Protobuf, TcpProtocol, Auth, EndPoints
from ctrader_open_api.messages.OpenApiCommonMessages_pb2 import *
from ctrader_open_api.messages.OpenApiMessages_pb2 import *
from ctrader_open_api.messages.OpenApiModelMessages_pb2 import *
from twisted.internet import reactor
import json
import datetime
import calendar
import keyring
import pandas as pd
import numpy as np

:0: UserWarning: You do not have a working installation of the service_identity module: 'No module named 'service_identity''.  Please install it from <https://pypi.python.org/pypi/service_identity> and make sure all of its dependencies are satisfied.  Without the service_identity module, Twisted can perform only rudimentary TLS client hostname verification.  Many valid certificate/hostname mappings may be rejected.


In [3]:
with open("credentials-dev.json") as f:
    credentials = json.load(f)
credentials['Secret'] = keyring.get_password("ctrader", credentials['ClientId'])

host = EndPoints.PROTOBUF_LIVE_HOST if credentials["HostType"].lower() == "live" else EndPoints.PROTOBUF_DEMO_HOST
client = Client(host, EndPoints.PROTOBUF_PORT, TcpProtocol)

In [4]:
PERIOD_MAP = {
    "M1":  ProtoOATrendbarPeriod.M1,
    "M2":  ProtoOATrendbarPeriod.M2,
    "M3":  ProtoOATrendbarPeriod.M3,
    "M4":  ProtoOATrendbarPeriod.M4,
    "M5":  ProtoOATrendbarPeriod.M5,
    "M10": ProtoOATrendbarPeriod.M10,
    "M15": ProtoOATrendbarPeriod.M15,
    "M30": ProtoOATrendbarPeriod.M30,
    "H1":  ProtoOATrendbarPeriod.H1,
    "H4":  ProtoOATrendbarPeriod.H4,
    "H12": ProtoOATrendbarPeriod.H12,
    "D1":  ProtoOATrendbarPeriod.D1,
    "W1":  ProtoOATrendbarPeriod.W1,
    "MN1": ProtoOATrendbarPeriod.MN1,
}

if period_str not in PERIOD_MAP:
    raise ValueError(f"Unknown period '{period_str}'. Valid options: {list(PERIOD_MAP.keys())}")

bar_period  = PERIOD_MAP[period_str]
total_weeks = num_chunks * weeks_per_chunk
output_path = f"../../data/{symbolName}_{period_str}_{total_weeks}weeks.csv"
print(f"Output will be saved to: {output_path}")

Output will be saved to: ../../data/XAUUSD_M1_520weeks.csv


In [5]:
dailyBars = []

def transformTrendbar(trendbar):
    openTime   = datetime.datetime.fromtimestamp(trendbar.utcTimestampInMinutes * 60, datetime.timezone.utc)
    openPrice  = (trendbar.low + trendbar.deltaOpen)  / 100000.0
    highPrice  = (trendbar.low + trendbar.deltaHigh)  / 100000.0
    lowPrice   =  trendbar.low                        / 100000.0
    closePrice = (trendbar.low + trendbar.deltaClose) / 100000.0
    return [openTime, openPrice, highPrice, lowPrice, closePrice, trendbar.volume]

In [6]:
def symbolsResponseCallback(result):
    print("\nSymbols received")
    symbols = Protobuf.extract(result)
    symbolsFilterResult = list(filter(lambda s: s.symbolName == symbolName, symbols.symbol))
    if len(symbolsFilterResult) == 0:
        raise Exception(f"No symbol matches '{symbolName}'")
    elif len(symbolsFilterResult) > 1:
        raise Exception(f"Multiple symbols match '{symbolName}': {symbolsFilterResult}")
    symbol = symbolsFilterResult[0]

    now = datetime.datetime.utcnow()
    requests = []
    for i in range(num_chunks):
        to_time   = now - datetime.timedelta(weeks=weeks_per_chunk * i)
        from_time = to_time - datetime.timedelta(weeks=weeks_per_chunk)
        request = ProtoOAGetTrendbarsReq()
        request.symbolId            = symbol.symbolId
        request.ctidTraderAccountId = credentials["AccountId"]
        request.period              = bar_period
        request.fromTimestamp       = int(calendar.timegm(from_time.utctimetuple())) * 1000
        request.toTimestamp         = int(calendar.timegm(to_time.utctimetuple()))   * 1000
        requests.append(request)

    def fetch_next(index):
        if index >= len(requests):
            print("\nAll chunks fetched")
            reactor.stop()
            return
        deferred = client.send(requests[index])
        def on_success(result):
            trendbars = Protobuf.extract(result)
            barsData = list(map(transformTrendbar, trendbars.trendbar))
            global dailyBars
            dailyBars.extend(barsData)
            print(f"\nFetched chunk {index+1}/{len(requests)}, bars: {len(barsData)}")
            fetch_next(index + 1)
        deferred.addCallbacks(on_success, onError)

    global dailyBars
    dailyBars.clear()
    fetch_next(0)

def accountAuthResponseCallback(result):
    print("\nAccount authenticated")
    request = ProtoOASymbolsListReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.includeArchivedSymbols = False
    deferred = client.send(request)
    deferred.addCallbacks(symbolsResponseCallback, onError)

def applicationAuthResponseCallback(result):
    print("\nApplication authenticated")
    request = ProtoOAAccountAuthReq()
    request.ctidTraderAccountId = credentials["AccountId"]
    request.accessToken = credentials["AccessToken"]
    deferred = client.send(request)
    deferred.addCallbacks(accountAuthResponseCallback, onError)

def onError(failure):
    print("\nMessage Error:", failure)

def disconnected(client, reason):
    print("\nDisconnected:", reason)

def onMessageReceived(client, message):
    if message.payloadType in [
        ProtoHeartbeatEvent().payloadType,
        ProtoOAAccountAuthRes().payloadType,
        ProtoOAApplicationAuthRes().payloadType,
        ProtoOASymbolsListRes().payloadType,
        ProtoOAGetTrendbarsRes().payloadType,
    ]:
        return
    print("\nMessage received:\n", Protobuf.extract(message))

def connected(client):
    print("\nConnected")
    request = ProtoOAApplicationAuthReq()
    request.clientId     = credentials["ClientId"]
    request.clientSecret = credentials["Secret"]
    deferred = client.send(request)
    deferred.addCallbacks(applicationAuthResponseCallback, onError)

client.setConnectedCallback(connected)
client.setDisconnectedCallback(disconnected)
client.setMessageReceivedCallback(onMessageReceived)

In [7]:
client.startService()
reactor.run()


Connected

Application authenticated

Account authenticated

Symbols received

Fetched chunk 1/520, bars: 6876

Fetched chunk 2/520, bars: 6887

Fetched chunk 3/520, bars: 6887

Fetched chunk 4/520, bars: 6947

Fetched chunk 5/520, bars: 6887

Fetched chunk 6/520, bars: 6738

Fetched chunk 7/520, bars: 6887

Fetched chunk 8/520, bars: 6887

Fetched chunk 9/520, bars: 6887

Fetched chunk 10/520, bars: 6739

Fetched chunk 11/520, bars: 6887

Fetched chunk 12/520, bars: 6887

Fetched chunk 13/520, bars: 5385

Fetched chunk 14/520, bars: 5317

Fetched chunk 15/520, bars: 6887

Fetched chunk 16/520, bars: 6887

Fetched chunk 17/520, bars: 6879

Fetched chunk 18/520, bars: 6608

Fetched chunk 19/520, bars: 6887

Fetched chunk 20/520, bars: 6888

Fetched chunk 21/520, bars: 6887

Fetched chunk 22/520, bars: 6827

Fetched chunk 23/520, bars: 6888

Fetched chunk 24/520, bars: 6889

Fetched chunk 25/520, bars: 6887

Fetched chunk 26/520, bars: 6887

Fetched chunk 27/520, bars: 6888

Fetched chu

In [8]:
df = pd.DataFrame(
    np.array(dailyBars),
    columns=['Time', 'Open', 'High', 'Low', 'Close', 'Volume']
).drop_duplicates().reset_index(drop=True)

for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[col] = pd.to_numeric(df[col])

In [9]:
df['Time'].describe()

count                             3513598
mean     2021-04-05 02:09:47.191506+00:00
min             2016-04-10 23:23:00+00:00
25%             2018-10-08 13:59:15+00:00
50%             2021-04-07 13:39:30+00:00
75%             2023-09-27 19:11:45+00:00
max             2026-03-29 23:22:00+00:00
Name: Time, dtype: object

In [10]:
df = df.sort_values('Time').drop_duplicates().reset_index(drop=True)
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")

Saved 3513598 rows to ../../data/XAUUSD_M1_520weeks.csv


In [11]:
df

,Time,Open,High,Low,Close,Volume
0,2016-04-10 23:23:00+00:00,1244.79,1244.79,1244.71,1244.71,4
1,2016-04-10 23:24:00+00:00,1244.80,1244.83,1244.63,1244.81,34
2,2016-04-10 23:25:00+00:00,1244.82,1244.82,1244.73,1244.80,18
3,2016-04-10 23:26:00+00:00,1244.79,1245.03,1244.61,1244.71,70
4,2016-04-10 23:27:00+00:00,1244.73,1244.77,1244.63,1244.77,28
...,...,...,...,...,...,...
3513593,2026-03-29 23:18:00+00:00,4482.05,4483.03,4481.02,4482.01,283
3513594,2026-03-29 23:19:00+00:00,4481.72,4481.93,4475.18,4475.53,290
3513595,2026-03-29 23:20:00+00:00,4475.54,4479.47,4475.29,4478.34,339
3513596,2026-03-29 23:21:00+00:00,4478.35,4479.34,4476.84,4477.71,312
